# wrangle / portfolio / allocation

**Intent**
- Takes `portfolio_allocation_volatility_pct`, in percent a year.
- Outputs `portfolio_allocation_risky_pct`, the risky share that maximises utility, capped at 100.
- Outputs `portfolio_allocation_riskadjusted_pct`, the risk-adjusted excess return in percent a year.
- Both come from one excess return and risk aversion, so they cannot disagree.
- The share is the Merton share, an upper bound.
- The risk-adjusted return puts a 100 percent equity portfolio and a 60/40 one on one scale.
- Acceptance holds the rmse of each output on `validation` and `backtest` to 0.01.

**Data**
- Personal data is git-ignored in `data/`, with synthetic placeholders committed.
- `training` is ten volatilities with both outputs, `validation` is five further, and `backtest` is ten across the range.
- `backtest` includes capped shares and is held back for acceptance.
- `golden` is volatilities from 5 to 60 percent, then a request with the input missing.

**Status**
- The excess return and the risk aversion are recovered from synthetic points.

**Future**
- Overall: the work still to do before the first release:
  - Add `query_postgres` to `workbench.data` and load the real series in place of the synthetic ones.
  - Replace the fit with the Merton share, `(mu - r) / (gamma * sigma^2)`, from `portfolio/return`, `economy/rates` and `portfolio/utility`.
  - Report the share and the risk-adjusted return at low, central and high risk aversion.
  - Treat the share as an upper bound, since it assumes constant returns, no income floor and no taxes.
  - Get excess return and volatility per asset class, leverage limits and the cash rate history.
  - Fix the limits against real data, then register and review before any release.

## Setup ([6.7.1](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#671-setup))

In [ ]:
from os.path import join

from workbench import setup_workspace

ws = setup_workspace("wrangle/portfolio/allocation", {"kind": "served"})


GOLDEN = "portfolio_allocation_golden"

## Data ([6.7.2](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#672-data))

In [ ]:
from workbench import data_load


def labelled(volatilities):
    adjusted = [4.0 - 0.5 * 3.0 * (volatility / 100.0) ** 2 * 100.0 for volatility in volatilities]
    shares = [min(100.0, 4.0 / 3.0 / (volatility / 100.0) ** 2) for volatility in volatilities]
    return {ws.inputs[0]: volatilities, ws.outputs[0]: [round(share, 6) for share in shares], ws.outputs[1]: [round(value, 6) for value in adjusted]}


training_pl, training_path = data_load(ws, "training", "portfolio_allocation_training", run=1, source="builder", build=lambda: labelled([5.0, 8.0, 10.0, 12.0, 15.0, 16.0, 18.0, 20.0, 22.0, 25.0]))

validation_pl, validation_path = data_load(ws, "validation", "portfolio_allocation_validation", run=1, source="builder", build=lambda: labelled([6.0, 9.0, 14.0, 19.0, 24.0]))

backtest_pl, backtest_path = data_load(ws, "backtest", "portfolio_allocation_backtest", run=1, source="builder", build=lambda: labelled([7.0, 11.0, 13.0, 17.0, 21.0, 23.0, 26.0, 28.0, 30.0, 32.0]))
print("training", training_pl.height, "validation", validation_pl.height, "backtest", backtest_pl.height)

## Train ([6.7.4](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#674-train))

In [ ]:
import numpy as np

variance = (training_pl[ws.inputs[0]].to_numpy() / 100.0) ** 2 * 100.0
slope, intercept = np.polyfit(variance, training_pl[ws.outputs[1]].to_numpy(), 1)
PARAMS = {"excess": round(float(intercept), 4), "gamma": round(float(-2.0 * slope), 4)}
print("fitted", PARAMS)
assert PARAMS == {"excess": 4.0, "gamma": 3.0}, "the fit did not recover the excess return and risk aversion"

## Evaluate ([6.7.5](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#675-evaluate))

In [ ]:
from workbench import evaluate_check, evaluate_measure, evaluate_predict

MODEL_TEMPLATE = """import pandas as pd
from mlflow.models import set_model
from mlflow.pyfunc import PythonModel

INPUT = {inputs[0]!r}
OUTPUT_RISKY = {outputs[0]!r}
OUTPUT_ADJUSTED = {outputs[1]!r}
EXCESS = {excess!r}
GAMMA = {gamma!r}


def risky_share(volatility):
    return 100.0 if volatility <= 0.0 else min(100.0, EXCESS / GAMMA / (volatility / 100.0) ** 2)


class Allocation(PythonModel):
    def predict(self, context, model_input: pd.DataFrame, params=None):
        volatilities = list(model_input[INPUT])
        return pd.DataFrame(
            {{
                OUTPUT_RISKY: [risky_share(volatility) for volatility in volatilities],
                OUTPUT_ADJUSTED: [EXCESS - 0.5 * GAMMA * (volatility / 100.0) ** 2 * 100.0 for volatility in volatilities],
            }}
        )


set_model(Allocation())
"""

model_path = join(ws.path, "model.py")
with open(model_path, "w") as model_file:
    model_file.write(MODEL_TEMPLATE.format(**PARAMS, inputs=ws.inputs, outputs=ws.outputs))

predict = evaluate_predict(model_path, "pyfunc", ws.signature)

LIMITS = {ws.outputs[0]: 0.01, ws.outputs[1]: 0.01}
SPLITS = (("validation", validation_pl), ("backtest", backtest_pl))


def acceptance_checks():
    answers = {split: predict({name: frame[name].to_list() for name in ws.inputs}) for split, frame in SPLITS}
    return [
        {"split": split, "output": output, "name": "rmse", "value": evaluate_measure(frame[output].to_list(), answers[split][output])["rmse"], "limit": LIMITS[output], "direction": "max"}
        for split, frame in SPLITS
        for output in ws.outputs
    ]


evaluation = evaluate_check(acceptance_checks, {"validation": [validation_path], "backtest": [backtest_path]})
for entry in evaluation["checks"]:
    print(entry["split"], entry["output"], entry["name"], "measured", entry["value"], "passed", entry["passed"])
print("acceptance", "passed" if evaluation["passed"] else "failed, the shipped model is outside the limit")

## Package ([6.7.6](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#676-package))

In [ ]:
from workbench import package_workspace


def golden_samples():
    xs = [5.0, 10.0, 12.2, 15.0, 20.0, 30.0, 60.0]
    answers = predict({ws.inputs[0]: xs})
    return {ws.inputs[0]: [*xs, None], **{output: [*answers[output], None] for output in ws.outputs}, "expect": ["ok"] * len(xs) + ["error"]}


golden = golden_samples()
package_workspace(
    ws,
    "pyfunc",
    golden,
    trained_on="synthetic risk-adjusted returns and risky shares, training split",
    inputs=[training_path],
    notes="least squares on the variance",
    implementation="mlflow.pyfunc model.py, requires mlflow, pandas and the standard library",
    model_path=model_path,
    evaluation=evaluation,
    sequence="across the range of the inputs",
)

## Register ([6.7.7](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#677-register))

In [ ]:
from workbench import register_workspace

register_workspace(ws)